#Introduction

Retrieval Augmented Generation (RAG) enhances large language models by retrieving relevant information before generating responses.
However, traditional vector-based RAG struggles with multi-hop reasoning and explicit relationships between concepts.

GraphRAG addresses this limitation by:

* extracting structured knowledge (entities and relations),

* storing it as a knowledge graph,

* combining semantic similarity with graph traversal.

This notebook demonstrates a complete GraphRAG pipeline using an organization-hosted LLM accessed via a custom base URL and API key.

# What This Notebook Contains

* Creation of a small but meaningful document corpus

* LLM-based knowledge triple extraction

* Knowledge graph construction

* Semantic embeddings of graph nodes

* Similarity-based seed selection

* Graph-based multi-hop retrieval

* Context generation for LLM reasoning

* Optional visualization and answer generation

* Clear observations explaining system behavior

#Step-1:Environment Setup

In [ ]:
!pip -q install networkx sentence-transformers openai matplotlib


#Step-2:Library Installation

In [ ]:
import os
import re
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from openai import OpenAI


#Step-3:Initialize Organization LLM Client

In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
OPENAI_BASE_URL = userdata.get("OPENAI_BASE_URL")
from openai import OpenAI

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL
)


#Step-4:Document Corpus Creation

In [ ]:
docs = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Neural networks are inspired by the human brain.",
    "Artificial intelligence is used in education for personalized learning."
]


#Step-5:LLM-Based Knowledge Triple Extraction

In [ ]:
def llm_extract_triples(text):
    prompt = f"""
Extract knowledge triples from the text.

Return one triple per line in the format:
(subject | relation | object)

Rules:
- Use short snake_case relations
- Do not explain anything
- Only output triples

Text:
{text}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": "You extract knowledge triples."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    triples = []
    output = response.choices[0].message.content.strip()

    for line in output.split("\n"):
        if "|" in line:
            s, r, o = [x.strip().lower() for x in line.split("|")]
            triples.append((s, r, o))

    return triples


#Step-6:Knowledge Graph Construction

In [ ]:
G = nx.DiGraph()

for doc in docs:
    triples = llm_extract_triples(doc)
    for h, r, t in triples:
        G.add_edge(h, t, relation=r, source=doc)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


#Step-7:Graph Node Embeddings

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

nodes = list(G.nodes())
node_embeddings = embed_model.encode(nodes, convert_to_numpy=True)


#Step-8:Similarity Metric Definition

In [ ]:
def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


#Step-9:Graph-Based Retrieval Function

In [ ]:
def graph_retrieve(query, top_k=2, hops=2):
    q_emb = embed_model.encode([query], convert_to_numpy=True)[0]

    sims = [cosine_sim(q_emb, emb) for emb in node_embeddings]
    top_idx = np.argsort(sims)[::-1][:top_k]
    seed_nodes = [nodes[i] for i in top_idx]

    expanded = set(seed_nodes)

    for _ in range(hops):
        new_nodes = set()
        for n in expanded:
            if n in G:
                new_nodes.update(G.successors(n))
                new_nodes.update(G.predecessors(n))
        expanded.update(new_nodes)

    subG = G.subgraph(expanded).copy()
    return seed_nodes, subG


#Step-10:Query Execution

In [ ]:
query = "How are neural networks related to artificial intelligence in education?"

seed_nodes, subG = graph_retrieve(query)

print("Seed Nodes:", seed_nodes)


#Step-11:Subgraph to Context Conversion
This cell converts the retrieved subgraph into a structured textual context
that can be passed to a language model for answer generation.


In [ ]:
def subgraph_to_context(subG):
    lines = []
    for u, v, d in subG.edges(data=True):
        lines.append(f"{u} --[{d['relation']}]--> {v}")
    return "\n".join(lines)

print(subgraph_to_context(subG))


#Graph Visualization

In [ ]:
plt.figure(figsize=(10,6))
pos = nx.spring_layout(subG, seed=42)
nx.draw(subG, pos, with_labels=True, node_size=2500)
edge_labels = nx.get_edge_attributes(subG, "relation")
nx.draw_networkx_edge_labels(subG, pos, edge_labels=edge_labels)
plt.show()


#Step-12:LLM-Based Answer Generation

In [ ]:
def generate_answer(query, context):
    prompt = f"""
Answer the question using the context.

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": "You answer using provided context only."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


### Observations
- LLM-based extraction creates richer and more accurate graphs than regex.
- Semantic similarity selects initial seed concepts.
- Graph expansion enables multi-hop reasoning.
- GraphRAG preserves relationships better than vector-only RAG.
